# Stat 220, Unit 4 Homework: Prediction and Choosing Predictors

`delivery_routes.csv` is 500 completed delivery routes. `minutes` is how long the
route took. The predictors that make sense are `stops`, `packages`, `miles`, `downtown`, and
`rain`. The company also records `van_age_years`, `dispatcher_rating`, and `month`.

Run the next two cells first. The first one installs what you need, and the second loads the
data that everything else here uses.

In [7]:
%pip install -q numpy pandas statsmodels scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Owner\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split

routes = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/delivery_routes.csv")
folds = KFold(5, shuffle=True, random_state=0)
routes.head()

,minutes,stops,packages,miles,downtown,rain,van_age_years,dispatcher_rating,month
0,254.3,49,105,55.6,1,0,8,3,7
1,153.5,24,71,32.0,0,0,5,5,2
2,450.9,73,226,55.4,1,1,7,4,4
3,387.1,71,175,73.2,1,0,1,1,12
4,111.2,12,18,32.1,0,1,7,1,11


**Problem 1.** *Planning tomorrow's schedule.* Tomorrow's route has 45 stops, 110 packages,
38 miles, goes downtown, and no rain is forecast.

Part a. Report the predicted minutes, the confidence interval, and the prediction interval. Say in one sentence what each of the two intervals is about.

In [9]:
fit = smf.ols("minutes ~ stops + packages + miles + downtown + rain", data=routes).fit()

new = pd.DataFrame({"stops": [45], "packages": [110], "miles": [38],
                    "downtown": [1], "rain": [0]})
print(fit.get_prediction(new).summary_frame(alpha=0.05).round(1))

    mean  mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  obs_ci_upper
0  273.9      2.1          269.7          278.1         229.9         317.8


The predicted time is 273.9 minutes. The 95% confidence interval is (269.7, 278.1) minutes, and the 95% prediction interval is (229.9, 317.8) minutes.

The confidence interval describes the uncertainty about the average time for routes with these predictor values, while the prediction interval describes the uncertainty about the time of one individual future route.



Part b. The dispatcher has to tell one driver when to expect to finish tomorrow. Which of the two intervals should they use, and why?

The dispatcher should use the prediction interval because they are trying to predict when one particular driver and route will finish.



Part c. The operations manager asks a different question: across all the downtown routes like this one, what is our average time? Which interval answers that one?

The confidence interval answers this question because the operations manager is asking about the average time across all downtown routes with these characteristics


**Problem 2.** *A route that looks ordinary.* A planner asks for a prediction for a route with
25 stops and 120 packages.

Part a. Each value is common on its own. Report how many completed routes are near each value, and how many are near both.

In [10]:
near_stops = routes.stops.between(20, 30)
near_packages = routes.packages.between(110, 130)

print("routes near this stop count   :", near_stops.sum())
print("routes near this package count:", near_packages.sum())
print("routes near both              :", (near_stops & near_packages).sum())
print("correlation between stops and packages:", round(routes.stops.corr(routes.packages), 2))

routes near this stop count   : 86
routes near this package count: 59
routes near both              : 0
correlation between stops and packages: 0.92


There are 86 routes near 25 stops, 59 routes near 120 packages, and 0 routes near both values.

The correlation between stops and packages is .92.


Part b. In three or four sentences, explain what your part a numbers mean for a prediction for this route, and what you would tell the planner who asked for it.

Even though 25 stops and 120 packages are each common values by themselves, the number of routes near both values is more important for this prediction. If very few routes have both characteristics, like in this example where 0 share them, then the model has less direct data for making a prediction at this combination of predictors. I would tell them that it is less well supported by the observed data than a prediction for a route with a more common combination of stops and packages.



**Problem 3.** *How wrong will it be?* A new depot has 50 completed routes so far, and the
manager wants one number for how far off a time estimate typically is.

Part a. Report both numbers, and say which one you would give the manager.

In [11]:
depot = routes.sample(50, random_state=3)
X = depot[["stops", "packages", "miles", "downtown", "rain"]]
y = depot["minutes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
m = LinearRegression().fit(X_train, y_train)

print("on the rows it was fitted on:", round(np.sqrt(((y_train - m.predict(X_train))**2).mean()), 1))
print("on the rows held back       :", round(np.sqrt(((y_test - m.predict(X_test))**2).mean()), 1))

on the rows it was fitted on: 22.9
on the rows held back       : 27.7


The RMSE on the rows used to fit the model is 22.9 minutes, while the RMSE on the held-back rows is 27.7 minutes.

I would give the manager the held-back/test-set RMSE of 27.7 minutes, because it better represents how far the model's predictions are typically off for new routes.



Part b. Explain in two or three sentences why those two numbers differ, and which way the difference would go if the depot had 500 routes instead of 50.

The training RMSE is usually smaller because the model was fitted using those same observations, so it has had a chance to adapt to their particular patterns and noise. With 500 routes instead of 50, I would expect the difference between training and test error to become smaller because the model would have much more data and would be less likely to overfit the particular 50 routes.



**Problem 4.** *Which predictors belong?* Still the 50-route depot.

Part a. Report the cross-validated error, the $R^2$, and the AIC for each set. Say which set you would ship.

In [12]:
sets = {
    "stops only": ["stops"],
    "the five that make sense": ["stops", "packages", "miles", "downtown", "rain"],
    "those five plus three junk": ["stops", "packages", "miles", "downtown", "rain",
                                   "van_age_years", "dispatcher_rating", "month"],
}
for name, cols in sets.items():
    mse = -cross_val_score(LinearRegression(), depot[cols], depot.minutes, cv=folds,
                           scoring="neg_mean_squared_error").mean()
    f = smf.ols("minutes ~ " + " + ".join(cols), data=depot).fit()
    print(f"{name:<28} CV error {np.sqrt(mse):5.1f}   R2 {f.rsquared:.4f}   AIC {f.aic:.1f}")

stops only                   CV error  38.1   R2 0.9165   AIC 503.7
the five that make sense     CV error  27.3   R2 0.9625   AIC 471.7
those five plus three junk   CV error  31.1   R2 0.9646   AIC 474.8


Stops only has a CV error of 38.1, a R2 of 0.9165 and a AIC of 503.7
the five that make sense has a CV error of 27.3, a R2 of 0.9625 and a AIC of 471.7
those five plus three junk has a CV error of 31.1, a R2 of 0.9646 and a AIC of 474.8

I would pick the five that make sense set as it has the lowest CV error and AIC

Part b. One set has the highest $R^2$ and is still not the set you would ship. Explain what $R^2$ is doing here in two or three sentences.

R² measures the proportion of variation in the observed response that is explained by the predictors on the data used to fit the model. Adding predictors can increase R² even when those predictors do not actually improve predictions on new data.



Part c. A colleague wants to add a squared term in `stops` on top of the five. Say what you would check before agreeing, and what you would expect with only 50 routes.

Before adding the squared stops term, I would check whether there is evidence that the relationship between stops and minutes is actually curved rather than approximately linear. With only 50 routes, I would be cautious because adding another term can make the model more sensitive to random variation and overfitting.



**Problem 5.** *What can and cannot be said.* No computer.

Part a. A colleague ran stepwise selection over 65 columns, kept the 11 with p-values under 0.05, and reports an $R^2$ of 0.96 with every predictor significant. Write the two or three sentences you would say in response.

I would not conclude that the model is genuinely this good just because the stepwise model has an R² of 0.96 and all 11 predictors are significant. Searching through 65 columns and selecting variables based on their p-values creates a large multiple-testing problem, so the reported p-values and R² can be overly optimistic.



Part b. A lasso fit sends `rain` to exactly zero. Does that show rain has no effect on how long a route takes? Explain.

No. Lasso setting rain to exactly zero means that, given the other predictors and the particular penalty chosen, the lasso model did not find enough additional predictive value from rain to keep its coefficient.



Part c. The model predicts well and `packages` has a large coefficient. The manager asks whether splitting the same deliveries into more packages would make routes take longer. Answer in two or three sentences.

No, a large coefficient for packages does not by itself show that splitting the same deliveries into more packages would cause a route to take longer. The coefficient describes an association while holding the other predictors fixed, but it does not establish that changing the number of packages would causally change the route time.

